# Buril — entrenamiento del limpiador de trazos (Google Colab)

Cuaderno **driver fino**: no copia la lógica, sólo llama a los scripts del repo.
Así, cuando `tools/entrenar_limpiar_trazos.py` cambie, este cuaderno ya lo usa en
el próximo `git pull`. La fuente de verdad son los `tools/*.py`.

Cadena: clonar repo → dataset (Plan B, Quick, Draw!) → entrenar → publicar en la
release `modelo-buril`, que es el buzón que lee `android-apk.yml`.

**Antes de empezar**: guardá en *Colab Secrets* (icono de llave 🔑, barra
izquierda) el secreto `GH_TOKEN` con un PAT fine-grained de permiso
*Contents: Read and write* sobre este repo. No lo escribas en una celda.

In [ ]:
# @title 1. Configuración y código del repo
REPO = "LuisAlejandro544/Dibujo-"   # usuario/repo
RAMA = "main"
TAG = "modelo-buril"                # release que guarda el modelo (buzón Colab -> APK)
LADO = "512"                        # lado del cache del dataset en px
POR_CATEGORIA = "100"               # dibujos máximos por categoría
EPOCAS = "30"
AUMENTOS = "1"
PARCHES = "16"                      # parches 64x64 por hoja

import os, subprocess

RUTA = "/content/Dibujo-"
if os.path.isdir(os.path.join(RUTA, ".git")):
    subprocess.run(["git", "fetch", "origin", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "checkout", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", RAMA], cwd=RUTA, check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", RAMA,
         f"https://github.com/{REPO}.git", RUTA], check=True)

os.chdir(RUTA)
# Colab ya trae TensorFlow; nos aseguramos de tener numpy + tensorflow.
subprocess.run(["pip", "install", "-q", "numpy", "tensorflow"], check=True)
print("Repositorio listo en", RUTA, "| rama", RAMA)

In [ ]:
# @title 2. Preparar el dataset (Plan B: Quick, Draw!, CC BY 4.0)
# Fases reanudables: si ya bajó/preprocesó, no repite la descarga.
!python3 tools/preparar_dataset_limpiar_trazos_planb.py todo --lado {LADO} --por-categoria {POR_CATEGORIA}
!python3 tools/preparar_dataset_limpiar_trazos_planb.py estado

In [ ]:
# @title 3. Entrenar Buril
NPZ = f".cache/limpiar_trazos_planb/procesado/pares_esq_lado{LADO}.npz"
SALIDA = ".cache/modelos/limpiar_trazos.tflite"
!python3 tools/entrenar_limpiar_trazos.py --pares {NPZ} --salida {SALIDA} \
    --epocas {EPOCAS} --aumentos {AUMENTOS} --parches-por-imagen {PARCHES} \
    --nombre Buril

In [ ]:
# @title 4. Publicar el modelo en la release `modelo-buril`
import json, os, requests
from google.colab import userdata

TOKEN = userdata.get("GH_TOKEN")
if not TOKEN:
    raise SystemExit("Falta el secreto GH_TOKEN en Colab (Contents: Read and write).")

CABECERAS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
API = f"https://api.github.com/repos/{REPO}"

def obtener_release():
    r = requests.get(f"{API}/releases/tags/{TAG}", headers=CABECERAS)
    if r.status_code == 200:
        return r.json()
    r = requests.post(f"{API}/releases", headers=CABECERAS, json={
        "tag_name": TAG,
        "name": "Modelo Buril",
        "body": ("Modelo `limpiar_trazos.tflite` (limpiador de trazos, float16). "
                 "Lo entrena Google Colab con "
                 "`tools/entrenar_limpiar_trazos_colab.ipynb`; `android-apk.yml` "
                 "lo baja y lo inyecta en el APK."),
    })
    r.raise_for_status()
    return r.json()

release = obtener_release()
for archivo in [SALIDA, os.path.splitext(SALIDA)[0] + ".json"]:
    nombre = os.path.basename(archivo)
    for viejo in requests.get(f"{API}/releases/{release['id']}/assets",
                              headers=CABECERAS).json():
        if viejo["name"] == nombre:
            requests.delete(f"{API}/releases/assets/{viejo['id']}", headers=CABECERAS)
    with open(archivo, "rb") as f:
        datos = f.read()
    subida = requests.post(
        f"https://uploads.github.com/repos/{REPO}/releases/{release['id']}/assets",
        headers={**CABECERAS, "Content-Type": "application/octet-stream"},
        params={"name": nombre}, data=datos)
    subida.raise_for_status()
    print(f"Publicado {nombre} ({len(datos)} bytes) en la release {TAG}")

with open(os.path.splitext(SALIDA)[0] + ".json", encoding="utf-8") as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))

## Siguiente paso

La release `modelo-buril` ya tiene el `.tflite`. El workflow
`android-apk.yml` lo baja e inyecta en `app/src/main/assets/modelos/` antes de
`assembleDebug`. Para armar el APK con este modelo, corré *Actions → "APK
Android (assembleDebug)" → Run workflow* (o abrí un PR contra `main`). Si la
release no existe todavía, el APK se arma igual y la app cae al adelgazado
algorítmico.